In [1]:
from transformers import AutoImageProcessor
import torch.optim as optim
import webdataset as wds
from modules import *

In [2]:
DATASET_PATH = "/home/austen/GeoDataset/dataset_sharded"
BATCH_SIZE = 8
WORKERS = 8
S2_LEVELS = range(3, 7)
S2_LEVEL_WEIGHTS = [1.0, 1.0, 1.0, 1.0]
LEARNING_RATE = 0.0001
PRETRAINED_MODEL_ID = "facebook/convnext-tiny-224"
EPOCHS = 1

In [3]:
processor = AutoImageProcessor.from_pretrained(PRETRAINED_MODEL_ID, use_fast=True)

dataset = GeoWebDataset(
    DATASET_PATH, 
    processor, 
    levels=S2_LEVELS, 
    shuffle=False
)

model = HierarchicalConvNeXt(
    pretrained_name = PRETRAINED_MODEL_ID,
    num_classes_per_level = dataset.num_classes_list
)

loader = wds.WebLoader(dataset.dataset, num_workers=WORKERS, pin_memory=True)

trainer = Trainer(model, loader)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    trainer.train_epoch(optimizer, weights=S2_LEVEL_WEIGHTS)

320it [00:18, 16.90it/s]


KeyboardInterrupt: 